# Deskriptive Statistik, Wahrscheinlichkeitsrechnung und statistische Inferenz

Dieses Notebook begleitet die Wiederholung und Erweiterung statistischer Grundlagen in der **Angewandten Datenanalyse**.

Als durchgehendes Beispiel dienen Würfelwürfe. Das Beispiel ist bewusst einfach gewählt: Die theoretischen Eigenschaften eines fairen Würfels sind bekannt und können später mit simulierten Daten verglichen werden.

## 1. Deskriptive Statistik am Beispiel von Würfelwürfen

Ein fairer sechsseitiger Würfel wird 100-mal simuliert. Die Ergebnisse werden anschließend mit den bekannten Werkzeugen der deskriptiven Statistik beschrieben.

Untersucht werden:

- Häufigkeitsverteilung
- arithmetisches Mittel
- Median und Modalwert
- Quantile
- Spannweite und Interquartilsabstand
- Varianz und Standardabweichung

### Bibliotheken importieren

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### 100 faire Würfelwürfe simulieren

Ein fester Zufalls-Seed sorgt dafür, dass bei jeder Ausführung dieselbe Folge von Zufallszahlen erzeugt wird. Dadurch bleiben die Ergebnisse der Demonstration reproduzierbar.

In [ ]:
# 100 faire Würfelwürfe erzeugen und in einem DataFrame speichern, rng: Random Number Generator
rng = np.random.default_rng(42)

n = 100
wuerfe = rng.integers(1, 7, size=n)

df_wuerfel = pd.DataFrame({
    "Augenzahl": wuerfe
})

df_wuerfel.head()

### Häufigkeitsverteilung der Würfelergebnisse

In [ ]:
# Absolute Häufigkeiten der simulierten Augenzahlen bestimmen
haeufigkeiten = df_wuerfel["Augenzahl"].value_counts().sort_index()
haeufigkeiten

In [ ]:
# Häufigkeitsverteilung als Balkendiagramm darstellen
haeufigkeiten.plot(kind="bar")

plt.xlabel("Augenzahl")
plt.ylabel("Absolute Häufigkeit")
plt.title("Verteilung von 100 Würfelwürfen")
plt.show()

Obwohl der Würfel fair ist, treten die sechs Augenzahlen in einer endlichen Serie nicht exakt gleich häufig auf. Zufällige Schwankungen sind daher auch bei einem bekannten Zufallsmechanismus zu erwarten.

Diese Beobachtung wird später für die statistische Inferenz wieder wichtig.

### Lage- und Streuungsparameter mit `describe()`

`describe()` liefert für eine numerische Variable mehrere zentrale Kennzahlen auf einmal:

- `count`: Anzahl der Beobachtungen
- `mean`: arithmetisches Mittel
- `std`: Standardabweichung
- `min`: Minimum
- `25%`: unteres Quartil
- `50%`: Median
- `75%`: oberes Quartil
- `max`: Maximum

**Hinweis:** `pandas` berechnet `std` in `describe()` standardmäßig mit dem Divisor $n-1$. Die auf der Folie eingeführte deskriptive Standardabweichung mit dem Divisor $n$ wird weiter unten explizit mit `ddof=0` berechnet.

In [ ]:
# Zentrale deskriptive Kennzahlen ausgeben
df_wuerfel["Augenzahl"].describe()

### Lageparameter

Die bekannten Lageparameter werden zusätzlich einzeln berechnet.

In [ ]:
# Arithmetisches Mittel, Median und Modalwert berechnen
mittelwert = df_wuerfel["Augenzahl"].mean()
median = df_wuerfel["Augenzahl"].median()
modalwert = df_wuerfel["Augenzahl"].mode()

print("Mittelwert:", mittelwert)
print("Median:", median)
print("Modalwert:", modalwert.tolist())

Für die simulierten 100 Würfe ergeben sich:

- Mittelwert: **3,64**
- Median: **4**
- Modalwert: **5**

Der empirische Mittelwert liegt damit nahe beim theoretischen Mittelwert eines fairen Würfels von 3,5, stimmt aber nicht exakt mit ihm überein.

### Quantile

Quantile teilen die geordneten Beobachtungen in vorgegebene Anteile. Besonders häufig werden das 25-%-, 50-%- und 75-%-Quantil betrachtet.

In [ ]:
# 25-%-, 50-%- und 75-%-Quantil berechnen
quantile = df_wuerfel["Augenzahl"].quantile([0.25, 0.5, 0.75])
quantile

Das 50-%-Quantil entspricht dem Median.

### Spannweite und Interquartilsabstand

Die Spannweite berücksichtigt Minimum und Maximum. Der Interquartilsabstand beschreibt dagegen die Breite der mittleren 50 % der Beobachtungen.

In [ ]:
# Spannweite und Interquartilsabstand berechnen
spannweite = (
    df_wuerfel["Augenzahl"].max()
    - df_wuerfel["Augenzahl"].min()
)

iqr = (
    df_wuerfel["Augenzahl"].quantile(0.75)
    - df_wuerfel["Augenzahl"].quantile(0.25)
)

print("Spannweite:", spannweite)
print("Interquartilsabstand:", iqr)

### Varianz und Standardabweichung

Auf der Folie wurden Varianz und Standardabweichung für die vorliegenden Daten mit dem Divisor $n$ definiert. Deshalb wird hier `ddof=0` verwendet.

`pandas.var()` und `pandas.std()` verwenden ohne weitere Angabe standardmäßig `ddof=1` und damit den Divisor $n-1$. Dieser Unterschied wird später im Zusammenhang mit Stichproben und statistischer Inferenz erneut aufgegriffen.

In [ ]:
# Deskriptive Varianz und Standardabweichung mit Divisor n berechnen
varianz = df_wuerfel["Augenzahl"].var(ddof=0)
standardabweichung = df_wuerfel["Augenzahl"].std(ddof=0)

print("Varianz:", varianz)
print("Standardabweichung:", standardabweichung)

### Zwischenfazit

Die 100 simulierten Würfelwürfe können mit den bekannten Werkzeugen der deskriptiven Statistik vollständig beschrieben werden:

- Häufigkeiten zeigen die beobachtete Verteilung.
- Lageparameter beschreiben typische bzw. zentrale Werte.
- Streuungsparameter beschreiben die Variabilität der beobachteten Werte.

Die Daten zeigen zugleich zufällige Schwankungen. Für deren systematische Beschreibung wird im nächsten Theorieblock die Wahrscheinlichkeitsrechnung wiederholt.

## 2. Wahrscheinlichkeitsrechnung

### 2.1 Zufallsvariable und Erwartungswert

Ein fairer Würfel dient weiterhin als durchgehendes Beispiel. Die theoretische Wahrscheinlichkeitsverteilung wird mit den beobachteten Würfelwürfen verglichen.

In [ ]:
# Theoretische Wahrscheinlichkeitsverteilung eines fairen Würfels

augenzahlen = np.arange(1, 7)
wahrscheinlichkeiten = np.repeat(1/6, 6)

plt.bar(augenzahlen, wahrscheinlichkeiten)

plt.xlabel("Augenzahl")
plt.ylabel("Wahrscheinlichkeit")
plt.title("Wahrscheinlichkeitsverteilung eines fairen Würfels")
plt.xticks(augenzahlen)
plt.ylim(0, 0.20)

plt.show()

In [ ]:
# Relative Häufigkeiten der beobachteten Würfelwürfe

relative_haeufigkeiten = (
    df_wuerfel["Augenzahl"]
    .value_counts(normalize=True)
    .sort_index()
)

relative_haeufigkeiten

In [ ]:
# Theoretische Wahrscheinlichkeiten und beobachtete relative Häufigkeiten vergleichen

vergleich = pd.DataFrame({
    "Theoretische Wahrscheinlichkeit": wahrscheinlichkeiten,
    "Relative Häufigkeit": relative_haeufigkeiten.reindex(augenzahlen).values
}, index=augenzahlen)

vergleich.plot(kind="bar")

plt.xlabel("Augenzahl")
plt.ylabel("Wahrscheinlichkeit / relative Häufigkeit")
plt.title("Theoretische und beobachtete Verteilung")
plt.axhline(1/6, linestyle="--")
plt.show()

In [ ]:
# Theoretischer Erwartungswert eines fairen Würfels
augenzahlen = np.arange(1, 7)
wahrscheinlichkeiten = np.repeat(1/6, 6)

erwartungswert = np.sum(augenzahlen * wahrscheinlichkeiten)

# Empirischer Mittelwert der simulierten Würfe
mittelwert = df_wuerfel["Augenzahl"].mean()

print("Erwartungswert:", erwartungswert)
print("Empirischer Mittelwert:", mittelwert)

### 2.2 Gesetz der großen Zahlen

In [ ]:
# 10.000 Würfelwürfe für die Demonstration des Gesetzes der großen Zahlen erzeugen

rng_gzg = np.random.default_rng(42)

n_max = 10_000
wuerfe_gzg = rng_gzg.integers(1, 7, size=n_max)

In [ ]:
# Laufenden Mittelwert nach jedem Würfelwurf berechnen

laufender_mittelwert = (
    np.cumsum(wuerfe_gzg)
    / np.arange(1, n_max + 1)
)

In [ ]:
# Annäherung des empirischen Mittelwerts an den Erwartungswert darstellen

plt.plot(
    np.arange(1, n_max + 1),
    laufender_mittelwert
)

plt.axhline(
    y=3.5,
    linestyle="--",
    label="Erwartungswert = 3,5"
)

plt.xlabel("Anzahl der Würfelwürfe")
plt.ylabel("Empirischer Mittelwert")
plt.title("Gesetz der großen Zahlen")
plt.legend()

plt.show()

In [ ]:
# Empirischen Mittelwert für ausgewählte Stichprobengrößen anzeigen

for n in [10, 100, 1_000, 10_000]:
    print(
        f"{n:>5} Würfe: "
        f"Mittelwert = {laufender_mittelwert[n - 1]:.3f}"
    )

### 2.3 Varianz und Standardabweichung

Der Erwartungswert beschreibt die Lage einer Wahrscheinlichkeitsverteilung.

Varianz und Standardabweichung beschreiben dagegen, wie stark die möglichen Werte um den Erwartungswert streuen.

Für eine diskrete Zufallsvariable gilt:

$$
\operatorname{Var}(X)
=
\sum_x (x-E(X))^2 \cdot P(X=x)
$$

und

$$
\sigma_X = \sqrt{\operatorname{Var}(X)}
$$

Für den fairen Würfel sind alle sechs Augenzahlen gleich wahrscheinlich.

In [ ]:
# Theoretische Varianz und Standardabweichung eines fairen Würfels berechnen

augenzahlen = np.arange(1, 7)
wahrscheinlichkeiten = np.repeat(1/6, 6)

erwartungswert = np.sum(augenzahlen * wahrscheinlichkeiten)

varianz_theoretisch = np.sum(
    (augenzahlen - erwartungswert) ** 2
    * wahrscheinlichkeiten
)

standardabweichung_theoretisch = np.sqrt(varianz_theoretisch)

print("Erwartungswert:", erwartungswert)
print("Varianz:", varianz_theoretisch)
print("Standardabweichung:", standardabweichung_theoretisch)

Die theoretische Standardabweichung des fairen Würfels beträgt damit ungefähr

$$
\sigma_X \approx 1{,}71.
$$

Wie beim Mittelwert kann dieser theoretische Wert mit den Kennzahlen der tatsächlich beobachteten Würfelwürfe verglichen werden.

In [ ]:
# Theoretische und empirische Streuung vergleichen

varianz_empirisch = df_wuerfel["Augenzahl"].var(ddof=0)
standardabweichung_empirisch = df_wuerfel["Augenzahl"].std(ddof=0)

print("Theoretische Varianz:", varianz_theoretisch)
print("Empirische Varianz:", varianz_empirisch)

print()

print("Theoretische Standardabweichung:", standardabweichung_theoretisch)
print("Empirische Standardabweichung:", standardabweichung_empirisch)

Der Unterschied entspricht der bereits bekannten Situation beim Mittelwert:

- $E(X)$, $\operatorname{Var}(X)$ und $\sigma_X$ beschreiben die **theoretische Wahrscheinlichkeitsverteilung**.
- $\bar{x}$, $s^2$ und $s$ beschreiben die **beobachteten Daten**.

Bei einer endlichen Stichprobe stimmen theoretische und empirische Kennzahlen daher im Allgemeinen nicht exakt überein.

### 2.4 Rechenregeln für Erwartungswert und Varianz

Nun werden zwei unabhängige faire Würfel betrachtet.

Für jeden Würfel gilt theoretisch:

$$
E(X)=3{,}5
$$

und

$$
\operatorname{Var}(X)=\frac{35}{12}\approx 2{,}92.
$$

Für die Summe zweier unabhängiger Zufallsvariablen gelten:

$$
E(X_1+X_2)=E(X_1)+E(X_2)
$$

und

$$
\operatorname{Var}(X_1+X_2)
=
\operatorname{Var}(X_1)+\operatorname{Var}(X_2).
$$

Damit besitzt die Augensumme zweier fairer Würfel den Erwartungswert 7 und die Varianz

$$
\frac{35}{6}\approx 5{,}83.
$$

Die theoretischen Ergebnisse werden nun durch eine Simulation überprüft.

In [ ]:
# Zwei unabhängige faire Würfel jeweils 100.000-mal werfen

rng_summe = np.random.default_rng(42)

n_simulationen = 100_000

wuerfel_1 = rng_summe.integers(1, 7, size=n_simulationen)
wuerfel_2 = rng_summe.integers(1, 7, size=n_simulationen)

summe = wuerfel_1 + wuerfel_2
summe

In [ ]:
# Empirische Mittelwerte der beiden Würfel und ihrer Summe berechnen

print("Mittelwert Würfel 1:", wuerfel_1.mean())
print("Mittelwert Würfel 2:", wuerfel_2.mean())
print("Mittelwert der Summe:", summe.mean())

print()
print("Theoretischer Erwartungswert eines Würfels:", 3.5)
print("Theoretischer Erwartungswert der Summe:", 7)

Die simulierten Mittelwerte liegen nahe bei den theoretischen Erwartungswerten.

Insbesondere zeigt sich:

$$
E(X_1+X_2)
=
E(X_1)+E(X_2)
=
3{,}5+3{,}5
=
7.
$$

In [ ]:
# Empirische Varianzen der beiden Würfel und ihrer Summe berechnen

varianz_1 = np.var(wuerfel_1, ddof=0)
varianz_2 = np.var(wuerfel_2, ddof=0)
varianz_summe = np.var(summe, ddof=0)

print("Varianz Würfel 1:", varianz_1)
print("Varianz Würfel 2:", varianz_2)
print("Summe der beiden Varianzen:", varianz_1 + varianz_2)

print()
print("Varianz der Augensumme:", varianz_summe)
print("Theoretische Varianz der Augensumme:", 35/6)

Auch für die Varianz bestätigt die Simulation die Rechenregel für unabhängige Zufallsvariablen:

$$
\operatorname{Var}(X_1+X_2)
=
\operatorname{Var}(X_1)
+
\operatorname{Var}(X_2).
$$

Wichtig:

**Die Varianzen addieren sich – nicht die Standardabweichungen.**

In [ ]:
# Verteilung der Augensumme darstellen

pd.Series(summe).value_counts(normalize=True).sort_index().plot(kind="bar")

plt.xlabel("Augensumme")
plt.ylabel("Relative Häufigkeit")
plt.title("Verteilung der Summe zweier Würfel")
plt.show()

### 2.5 Normalverteilung

Als Beispiel wird eine normalverteilte Zufallsvariable betrachtet:

$$
X \sim N(100,15)
$$

Damit gilt:

- Erwartungswert: $\mu=100$
- Standardabweichung: $\sigma=15$

Gesucht ist die Wahrscheinlichkeit

$$
P(85 \leq X \leq 115).
$$

Bei einer stetigen Zufallsvariable gilt für einen einzelnen Wert:

$$
P(X=x)=0.
$$

Wahrscheinlichkeiten werden daher Intervallen zugeordnet. In der **Dichtefunktion** entsprechen sie Flächen unter der Kurve.

In [ ]:
# Normalverteilung mit Erwartungswert 100 und Standardabweichung 15 definieren

from scipy.stats import norm

mu = 100
sigma = 15
a = 85
b = 115

x = np.linspace(mu - 4*sigma, mu + 4*sigma, 1000)

dichte = norm.pdf(x, loc=mu, scale=sigma)
verteilung = norm.cdf(x, loc=mu, scale=sigma)

F_a = norm.cdf(a, loc=mu, scale=sigma)
F_b = norm.cdf(b, loc=mu, scale=sigma)

### Dichtefunktion und Verteilungsfunktion

Die **Dichtefunktion** beschreibt die Form der Verteilung. Ihre Höhe an einer bestimmten Stelle ist keine Wahrscheinlichkeit.

Die **Verteilungsfunktion**

$$
F(x)=P(X\leq x)
$$

gibt die Wahrscheinlichkeit an, dass die Zufallsvariable höchstens den Wert $x$ annimmt.

Für ein Intervall gilt:

$$
P(a\leq X\leq b)=F(b)-F(a).
$$

Für das Beispiel wird die Wahrscheinlichkeit $P(85\leq X\leq115)$ in beiden Darstellungen sichtbar gemacht.

In [ ]:
# Dichtefunktion und Verteilungsfunktion nebeneinander darstellen

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Dichtefunktion
ax[0].plot(x, dichte)

bereich = (x >= a) & (x <= b)
ax[0].fill_between(
    x[bereich],
    dichte[bereich],
    alpha=0.4
)

ax[0].axvline(a, linestyle="--")
ax[0].axvline(b, linestyle="--")
ax[0].set_xlabel("x")
ax[0].set_ylabel("Dichte")
ax[0].set_title("Dichtefunktion")

# Verteilungsfunktion
ax[1].plot(x, verteilung)
ax[1].scatter([a, b], [F_a, F_b])
ax[1].axvline(a, linestyle="--")
ax[1].axvline(b, linestyle="--")
ax[1].axhline(F_a, linestyle=":")
ax[1].axhline(F_b, linestyle=":")
ax[1].set_xlabel("x")
ax[1].set_ylabel("F(x)")
ax[1].set_title("Verteilungsfunktion")
ax[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

print("F(85) =", F_a)
print("F(115) =", F_b)
print("P(85 <= X <= 115) =", F_b - F_a)

Die Wahrscheinlichkeit ergibt sich aus der Differenz der beiden Werte der Verteilungsfunktion:

$$
P(85\leq X\leq115)
=
F(115)-F(85)
\approx 0{,}683.
$$

Damit liegen bei dieser Normalverteilung ungefähr **68,3 %** der Werte innerhalb einer Standardabweichung um den Erwartungswert.

### Lineare Transformation und Standardisierung

Eine lineare Transformation einer Zufallsvariable hat die Form

$$
Y=aX+b.
$$

Für Erwartungswert und Varianz gelten die bereits bekannten Rechenregeln:

$$
E(Y)=aE(X)+b
$$

und

$$
\operatorname{Var}(Y)=a^2\operatorname{Var}(X).
$$

Eine besonders wichtige lineare Transformation ist die **Standardisierung**:

$$
Z=\frac{X-\mu}{\sigma}.
$$

Dabei wird zunächst der Erwartungswert $\mu$ abgezogen und anschließend durch die Standardabweichung $\sigma$ geteilt.

Für die standardisierte Zufallsvariable gilt:

$$
E(Z)=0
$$

und

$$
\operatorname{Var}(Z)=1.
$$

Ist $X$ normalverteilt, dann ist auch $Z$ normalverteilt:

$$
Z\sim N(0,1).
$$

Diese Verteilung heißt **Standardnormalverteilung**.

Ein konkreter Wert

$$
z=\frac{x-\mu}{\sigma}
$$

gibt an, wie viele Standardabweichungen der beobachtete Wert $x$ vom Erwartungswert entfernt liegt.

- $z<0$: Wert liegt unterhalb des Erwartungswerts
- $z=0$: Wert entspricht dem Erwartungswert
- $z>0$: Wert liegt oberhalb des Erwartungswerts

In [ ]:
# Die Grenzen 85 und 115 standardisieren

z_a = (a - mu) / sigma
z_b = (b - mu) / sigma

print("z-Wert für 85:", z_a)
print("z-Wert für 115:", z_b)

Für das Beispiel gilt damit:

$$
85 \longrightarrow z=-1
$$

und

$$
115 \longrightarrow z=1.
$$

Die ursprüngliche Wahrscheinlichkeit kann daher äquivalent mit der Standardnormalverteilung berechnet werden:

$$
P(85\leq X\leq115)
=
P(-1\leq Z\leq1).
$$

Die Standardisierung verändert also nicht die gesuchte Wahrscheinlichkeit. Sie überführt die Zufallsvariable lediglich auf eine gemeinsame Skala mit Erwartungswert 0 und Standardabweichung 1.

In [ ]:
# Wahrscheinlichkeit über die Standardnormalverteilung berechnen

wahrscheinlichkeit_original = (
    norm.cdf(b, loc=mu, scale=sigma)
    - norm.cdf(a, loc=mu, scale=sigma)
)

wahrscheinlichkeit_standardisiert = (
    norm.cdf(z_b)
    - norm.cdf(z_a)
)

print("Originalverteilung:", wahrscheinlichkeit_original)
print("Standardnormalverteilung:", wahrscheinlichkeit_standardisiert)

### Quantile mit Python

Die Funktion `norm.cdf()` liefert zu einem Wert $x$ die kumulierte Wahrscheinlichkeit $F(x)$.

Die Umkehrfunktion `norm.ppf()` beantwortet die umgekehrte Frage:

**Welcher Wert $x$ gehört zu einer vorgegebenen kumulierten Wahrscheinlichkeit?**

Damit übernimmt Python die Aufgabe, für die früher Tabellen der Standardnormalverteilung verwendet wurden.

In [ ]:
# 95-%-Quantil der Beispielverteilung bestimmen

quantil_95 = norm.ppf(0.95, loc=mu, scale=sigma)
z_quantil_95 = norm.ppf(0.95)

print("95-%-Quantil der Verteilung:", quantil_95)
print("Zugehöriger z-Wert:", z_quantil_95)

### Zwischenfazit

Für die Normalverteilung sind insbesondere folgende Zusammenhänge wichtig:

- Wahrscheinlichkeiten entsprechen Flächen unter der Dichtefunktion.
- Die Verteilungsfunktion liefert kumulierte Wahrscheinlichkeiten.
- Durch Standardisierung wird eine normalverteilte Zufallsvariable auf die Standardnormalverteilung transformiert.
- Der $z$-Wert beschreibt den Abstand eines Wertes vom Erwartungswert in Einheiten der Standardabweichung.
- `norm.cdf()` und `norm.ppf()` ersetzen die praktische Arbeit mit Tabellen der Standardnormalverteilung.

Diese Grundlagen werden für die Verteilung von Stichprobenmittelwerten und den zentralen Grenzwertsatz benötigt.

### 2.6 Zentraler Grenzwertsatz und Stichprobenmittelwert

Der Stichprobenmittelwert ist die Summe der Beobachtungen geteilt durch die Stichprobengröße:

$$
\bar X=\frac{S_n}{n}
$$

mit

$$
S_n=X_1+\dots+X_n.
$$

Für die Summe unabhängiger, identisch verteilter Zufallsvariablen gilt:

$$
E(S_n)=n\mu
$$

und

$$
\sigma(S_n)=\sqrt{n}\,\sigma.
$$

Daraus folgt für den Stichprobenmittelwert:

$$
E(\bar X)
=
\frac{1}{n}E(S_n)
=
\mu
$$

und

$$
\sigma(\bar X)
=
\frac{1}{n}\sigma(S_n)
=
\frac{\sigma}{\sqrt n}.
$$

Für hinreichend großes $n$ ist die Verteilung der Stichprobenmittelwerte nach dem zentralen Grenzwertsatz näherungsweise normalverteilt:

$$
\bar X
\approx
N\left(\mu,\frac{\sigma}{\sqrt n}\right).
$$

Der Mittelwert bleibt bei $\mu$ zentriert. Seine Streuung wird mit wachsender Stichprobengröße kleiner.

#### Vom einzelnen Würfelwurf zum Stichprobenmittelwert

Ein einzelner fairer Würfelwurf ist diskret gleichverteilt.

Nun werden jeweils **zwei unabhängige Würfelwürfe** zu einer Stichprobe zusammengefasst und daraus der Mittelwert berechnet:

$$
\bar X=\frac{X_1+X_2}{2}.
$$

Die möglichen Stichprobenmittelwerte liegen zwischen 1 und 6 in Schritten von 0,5.

Die folgende Darstellung vergleicht die Verteilung eines einzelnen Würfelwurfs mit der Verteilung der Mittelwerte aus jeweils zwei Würfen.

In [ ]:
# Ursprüngliche Verteilung und Verteilung der Mittelwerte aus je zwei Würfen vergleichen

rng_vergleich = np.random.default_rng(42)

anzahl_stichproben = 100_000

# Einzelne Würfelwürfe
einzelwuerfe = rng_vergleich.integers(
    1, 7,
    size=anzahl_stichproben
)

# Jeweils zwei unabhängige Würfelwürfe
zwei_wuerfe = rng_vergleich.integers(
    1, 7,
    size=(anzahl_stichproben, 2)
)

# Mittelwert der jeweils zwei Würfe
mittelwerte_2 = zwei_wuerfe.mean(axis=1)

In [ ]:
# Verteilungen nebeneinander darstellen

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Einzelner Würfelwurf
werte, haeufigkeiten = np.unique(
    einzelwuerfe,
    return_counts=True
)

ax[0].bar(
    werte,
    haeufigkeiten / anzahl_stichproben
)

ax[0].set_xlabel("Augenzahl")
ax[0].set_ylabel("Relative Häufigkeit")
ax[0].set_title("Einzelner Würfelwurf")
ax[0].set_xticks(np.arange(1, 7))
ax[0].set_ylim(0, 0.35)

# Mittelwert aus zwei Würfen
werte, haeufigkeiten = np.unique(
    mittelwerte_2,
    return_counts=True
)

ax[1].bar(
    werte,
    haeufigkeiten / anzahl_stichproben,
    width=0.4
)

ax[1].set_xlabel("Stichprobenmittelwert")
ax[1].set_ylabel("Relative Häufigkeit")
ax[1].set_title("Mittelwert aus jeweils zwei Würfen")
ax[1].set_xticks(np.arange(1, 6.1, 0.5))
ax[1].set_ylim(0, 0.35)

plt.tight_layout()
plt.show()

Bereits bei Stichproben mit nur zwei Beobachtungen verändert sich die Verteilung deutlich:

- Der einzelne Würfelwurf ist gleichverteilt.
- Die Stichprobenmittelwerte konzentrieren sich stärker um den Erwartungswert $3{,}5$.
- Extreme Mittelwerte treten seltener auf als Werte nahe der Mitte.

Für einen einzelnen Würfelwurf gilt:

$$
\sigma(X)=\sqrt{\frac{35}{12}}\approx 1{,}71.
$$

Für den Mittelwert aus zwei unabhängigen Würfen gilt:

$$
\sigma(\bar X)
=
\frac{\sigma}{\sqrt{2}}
\approx
\frac{1{,}71}{\sqrt{2}}
\approx 1{,}21.
$$

Die Verteilung der Stichprobenmittelwerte ist damit bereits bei $n=2$ deutlich weniger gestreut als die Verteilung einzelner Würfelwürfe.

Mit zunehmender Stichprobengröße wird dieser Effekt stärker.

#### Größere Stichproben: Mittelwerte aus jeweils 30 Würfen

Mit zunehmender Stichprobengröße konzentrieren sich die Stichprobenmittelwerte immer stärker um den Erwartungswert.

Für $n=30$ beträgt die theoretische Standardabweichung der Stichprobenmittelwerte

$$
\sigma(\bar X)=\frac{\sigma}{\sqrt{30}}.
$$

Die folgende Simulation vergleicht diesen theoretischen Wert mit der empirischen Streuung vieler Stichprobenmittelwerte.

In [ ]:
# Erwartungswert und Standardabweichung eines einzelnen fairen Würfels

mu = 3.5
sigma = np.sqrt(35 / 12)

n = 30

sigma_mittelwert = sigma / np.sqrt(n)

print("Erwartungswert eines Würfelwurfs:", mu)
print("Standardabweichung eines Würfelwurfs:", sigma)
print()
print("Erwartungswert des Stichprobenmittelwerts:", mu)
print("Standardabweichung des Stichprobenmittelwerts:", sigma_mittelwert)

In [ ]:
# Viele Stichproben mit jeweils 30 Würfen simulieren

rng_mittelwerte = np.random.default_rng(42)

anzahl_stichproben = 10_000
n = 30

stichproben = rng_mittelwerte.integers(
    1, 7,
    size=(anzahl_stichproben, n)
)

stichprobenmittelwerte = stichproben.mean(axis=1)

print("Empirischer Mittelwert der Stichprobenmittelwerte:",
      stichprobenmittelwerte.mean())

print("Empirische Standardabweichung der Stichprobenmittelwerte:",
      stichprobenmittelwerte.std(ddof=0))

print("Theoretische Standardabweichung:",
      sigma / np.sqrt(n))

In [ ]:
# Verteilung der Stichprobenmittelwerte darstellen

plt.hist(
    stichprobenmittelwerte,
    bins=30,
    density=True,
    alpha=0.6
)

x = np.linspace(
    stichprobenmittelwerte.min(),
    stichprobenmittelwerte.max(),
    500
)

normaldichte = norm.pdf(
    x,
    loc=mu,
    scale=sigma / np.sqrt(n)
)

plt.plot(x, normaldichte)

plt.xlabel("Stichprobenmittelwert")
plt.ylabel("Dichte")
plt.title("Verteilung der Mittelwerte aus jeweils 30 Würfelwürfen")

plt.show()

Die Simulation zeigt zwei zentrale Eigenschaften der Stichprobenmittelwerte:

- Die Verteilung ist um $\mu=3{,}5$ zentriert.
- Die Streuung beträgt nur noch ungefähr $\sigma/\sqrt n$.
- Obwohl ein einzelner Würfelwurf nicht normalverteilt ist, ist die Verteilung der Mittelwerte bei $n=30$ bereits näherungsweise normalverteilt.

Damit liefert der zentrale Grenzwertsatz die Grundlage dafür, aus Stichproben Aussagen über unbekannte Parameter einer Grundgesamtheit abzuleiten.